# 0. Imports

In [1]:
#!pip install -qq ipython numpy pandas scikit-learn statsmodels xgboost torch

In [2]:
import sys
import warnings
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

import sklearn
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, root_mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import torch
from torch import nn
from torch.utils.data import Dataset, TensorDataset, DataLoader

In [3]:
!python --version
print("numpy", np.__version__)
print("pandas", pd.__version__)
print("scikit-learn", sklearn.__version__)
print("torch", torch.__version__)

Python 3.10.18
numpy 2.2.6
pandas 2.3.3
scikit-learn 1.7.2
torch 2.9.1


In [4]:
x = torch.randn(32, 24, 13)
lstm = nn.LSTM(13, 32, batch_first=True)
y, _ = lstm(x)
print("OK, y.shape =", y.shape)

OK, y.shape = torch.Size([32, 24, 32])


In [5]:
# Extract all datasets from /data/ into a nested dictionary.
DFDICT = {
    subdir.name: {
        csv.stem: pd.read_csv(csv, dtype="string", low_memory=False)
        for csv in subdir.glob("*.csv")
    }
    for subdir in (Path.cwd() / "data").iterdir() if subdir.is_dir()
}

# 1. Preprocessing

### 1.1 Select and Join Datasets

In [6]:
# Here we select what data to use from what's available.

# Specify a set of columns from each dataset for selection
weatherCols = [
    "DATE",
    "HourlyDryBulbTemperature",
    "HourlyRelativeHumidity",
    "HourlyWindSpeed",
    "HourlyStationPressure",
    "HourlyVisibility"
]
electricityCols = [
    "UTC time",
    "Hour",
    "Demand",
]

# Select all 2024 weather and electrical data from those columns
DFW = DFDICT['weather']['MIA_2024'][weatherCols]
DFE = DFDICT['electricity']['FPL'][electricityCols]
DFE = DFE.iloc[ (i := 24 * (8*365 + 2 + 31*4 + 30*2)) : i + 24*366].copy()

In [7]:
# Eletrical grid observations are ALWAYS every hour on the hour.  
# Weather observations are temporally much messier.
# The output below shows the distribution of temporal differences 
#  across all 2024 weather observations.

_df = pd.DataFrame({"t": pd.to_datetime(DFW["DATE"])})
_df["delta t"] = (
    _df["t"].diff()
    .apply(lambda x: "" if pd.isna(x) else str(x).split()[-1])
)

print("Number of rows: ", len(_df), end='\n\n')
print(_df.head(10), end="\n\n")
with pd.option_context("display.max_rows", None):
    print(_df["delta t"].value_counts().sort_index().reset_index(name="Count"))

Number of rows:  13105

                    t   delta t
0 2024-01-01 00:53:00          
1 2024-01-01 01:00:00  00:07:00
2 2024-01-01 01:53:00  00:53:00
3 2024-01-01 02:53:00  01:00:00
4 2024-01-01 03:53:00  01:00:00
5 2024-01-01 04:00:00  00:07:00
6 2024-01-01 04:53:00  00:53:00
7 2024-01-01 05:53:00  01:00:00
8 2024-01-01 06:53:00  01:00:00
9 2024-01-01 07:00:00  00:07:00

     delta t  Count
0                 1
1   00:00:00     17
2   00:01:00     16
3   00:02:00    186
4   00:03:00     72
5   00:04:00     48
6   00:05:00     61
7   00:06:00    393
8   00:07:00   2820
9   00:08:00     40
10  00:09:00     42
11  00:10:00     52
12  00:11:00     54
13  00:12:00     41
14  00:13:00     45
15  00:14:00     42
16  00:15:00     38
17  00:16:00     36
18  00:17:00     50
19  00:18:00     35
20  00:19:00     35
21  00:20:00     36
22  00:21:00     45
23  00:22:00     38
24  00:23:00     33
25  00:24:00     25
26  00:25:00     28
27  00:26:00     35
28  00:27:00     30
29  00:28:00     27
30 

In [8]:
# Each dataset's time columns is replace by with another in local time
#  which is UTC-5 (the weather data needs no time adjustment).
_dfe, _dfw = (
    DFE.drop(columns=["UTC time"])
       .assign(t=pd.to_datetime(DFE["UTC time"]) - pd.Timedelta(hours=5)),
    DFW.drop(columns=["DATE"])
       .assign(t=pd.to_datetime(DFW["DATE"]))
)

# Merge both dataframes, joining records on their time measurement.
# CRITICALLY, the nearest weather obversation to each hour is naively taken.
# THIS IS BAD AND NEEDS TO BE CHANGED.
DF = pd.merge_asof(_dfe, _dfw, on="t", direction="nearest")
display(DF)

,Hour,Demand,t,HourlyDryBulbTemperature,HourlyRelativeHumidity,HourlyWindSpeed,HourlyStationPressure,HourlyVisibility
0,24,"11,815",2024-01-01 00:00:00,57,81,5,30.20,7.00
1,1,"11,254",2024-01-01 01:00:00,57,81,5,30.20,6.84
2,2,"11,109",2024-01-01 02:00:00,57,81,0,30.19,6.00
3,3,"10,861",2024-01-01 03:00:00,56,87,0,30.18,6.00
4,4,"10,736",2024-01-01 04:00:00,55,87,3,30.17,9.94
...,...,...,...,...,...,...,...,...
8779,19,"16,796",2024-12-31 19:00:00,75,84,0,29.94,9.94
8780,20,"15,515",2024-12-31 20:00:00,74,88,3,<NA>,10.00
8781,21,"14,348",2024-12-31 21:00:00,74,88,0,<NA>,10.00
8782,22,"13,391",2024-12-31 22:00:00,73,90,5,29.98,9.94


In [9]:
# # Count missing values, list unique weather data values.

# print(DF.shape)
# print()

# for c in [*electricityCols[1:], *weatherCols[1:]]:
#     print(f"{c:<25} {(DF[c].isna()).sum()}")
# print()

# for c in weatherCols[1:]:
#     print(f"{c}: Unique Values")
#     print(DF[c].unique().tolist())
#     print()

### 1.2 Convert Datatypes and Construct Features

In [10]:
# Convert to numeric dtypes and impute missing values.
# CRITICALLY, the nearest observation is imputed for each missing value again.
# THIS IS BAD AND NEEDS TO BE CHANGED.

DF["HourlyDryBulbTemperature"] = DF["HourlyDryBulbTemperature"].str[:2]
for c in weatherCols[1:]:
    converted = pd.to_numeric(DF[c], errors="coerce") 
    intFlag = (converted.dropna() % 1 == 0).all()
    imputed = converted.interpolate(method="nearest").ffill().bfill()
    DF[c] = imputed.astype("uint8" if intFlag else "float32")

converted = pd.to_numeric(DF["Demand"].str.replace(",", "", regex=False), errors="coerce")
DF["Demand"] = converted.interpolate(method="nearest").fillna(0).astype("uint16")
DF["Hour"] = pd.to_numeric(DF["Hour"]).astype("uint8")

display(DF)

,Hour,Demand,t,HourlyDryBulbTemperature,HourlyRelativeHumidity,HourlyWindSpeed,HourlyStationPressure,HourlyVisibility
0,24,11815,2024-01-01 00:00:00,57,81,5,30.200001,7.00
1,1,11254,2024-01-01 01:00:00,57,81,5,30.200001,6.84
2,2,11109,2024-01-01 02:00:00,57,81,0,30.190001,6.00
3,3,10861,2024-01-01 03:00:00,56,87,0,30.180000,6.00
4,4,10736,2024-01-01 04:00:00,55,87,3,30.170000,9.94
...,...,...,...,...,...,...,...,...
8779,19,16796,2024-12-31 19:00:00,75,84,0,29.940001,9.94
8780,20,15515,2024-12-31 20:00:00,74,88,3,29.940001,10.00
8781,21,14348,2024-12-31 21:00:00,74,88,0,29.980000,10.00
8782,22,13391,2024-12-31 22:00:00,73,90,5,29.980000,9.94


In [11]:
# Make sure the column name is *exactly* this:
col = "HourlyStationPressure"

s = pd.to_numeric(DF[col], errors="coerce")
print("Before: NaNs in HourlyStationPressure =", s.isna().sum())

s = s.interpolate(method="nearest").ffill().bfill()
print("After interpolate+ffill+bfill: NaNs =", s.isna().sum())

DF[col] = s.astype("float32")
print("In DF: NaNs in HourlyStationPressure =", DF[col].isna().sum())

Before: NaNs in HourlyStationPressure = 1
After interpolate+ffill+bfill: NaNs = 0
In DF: NaNs in HourlyStationPressure = 0


In [12]:
# Construct additional features from the selected columns

# AUTOREGRESSIVE FEATURES
# In this case each include each of the previous 3 hrs of measured demand
#  along with the previous day and week (24 and 168 hrs).
# ! The first h rows are blank for these features - here we backfill them. CHANGE
# ! Many more should be included, especially for the DL models.
lag_hours = [1, 2, 3, 24, 168]
for h in lag_hours:
    DF[f"lag_{h}"] = DF["Demand"].shift(h).bfill().astype("float32")
lagFeats = [f"lag_{h}" for h in lag_hours]

# CALENDAR FEATURES
# For now, just trig transformations on the hour of day and a one-hot encoding for weekends.
# ! Flags can be included for each day of the week, each month, holidays, etc.
DF["sin(h)"] = np.sin(2 * np.pi * DF["Hour"] / 24.0).astype("float32")
DF["cos(h)"] = np.cos(2 * np.pi * DF["Hour"] / 24.0).astype("float32")
DF["is_weekend"] = (DF["t"].dt.dayofweek >= 5).astype("int8")
DF = DF.drop(columns=["Hour"])
calendarFeats = ["sin(h)", "cos(h)", "is_weekend"]

# WEATHER FEATURES
# We don't extract any new features yet for this category.
weatherFeats = weatherCols[1:]  # skip DATE

# Define other convenient feature variables.
allFeats = lagFeats + calendarFeats + weatherFeats
target = "Demand"

# Drop rows missing any needed inputs (initial lags, any NAs)
DFmodel = DF.sort_values("t").reset_index(drop=True)
print("Model DF shape:", DFmodel.shape)

nan_count = DFmodel.isna().sum()
print(nan_count)

Model DF shape: (8784, 15)
Demand                      0
t                           0
HourlyDryBulbTemperature    0
HourlyRelativeHumidity      0
HourlyWindSpeed             0
HourlyStationPressure       0
HourlyVisibility            0
lag_1                       0
lag_2                       0
lag_3                       0
lag_24                      0
lag_168                     0
sin(h)                      0
cos(h)                      0
is_weekend                  0
dtype: int64


### 1.3 Data Splits

In [13]:
# Split out last 5 weeks of data for testing.

nTestDays = 35
testStartTime = DFmodel["t"].max() - pd.Timedelta(days=nTestDays)

DFtrain = DFmodel[DFmodel["t"] <  testStartTime].copy()
DFtest  = DFmodel[DFmodel["t"] >= testStartTime].copy()

X_train = DFtrain[allFeats].to_numpy(dtype=np.float32)
y_train = DFtrain[target].to_numpy(dtype=np.float32)
X_test  = DFtest[allFeats].to_numpy(dtype=np.float32)
y_test  = DFtest[target].to_numpy(dtype=np.float32)

print("Train size:  ", len(DFtrain))
print("Test size:   ", len(DFtest))
print("Train range: ", DFtrain["t"].min(), " - ", DFtrain["t"].max())
print("Test range:  ", DFtest["t"].min(),  " - ", DFtest["t"].max())

Train size:   7943
Test size:    841
Train range:  2024-01-01 00:00:00  -  2024-11-26 22:00:00
Test range:   2024-11-26 23:00:00  -  2024-12-31 23:00:00


In [14]:
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test: ", X_test.shape,  "y_test: ", y_test.shape)

X_train: (7943, 13) y_train: (7943,)
X_test:  (841, 13) y_test:  (841,)


# 2. LSTM

In [15]:
seq_len = 24  # past 24 hours → next-hour demand

# --- scale X and y ---
x_scaler = StandardScaler()
X_train_s = x_scaler.fit_transform(X_train)
X_test_s  = x_scaler.transform(X_test)

y_scaler = StandardScaler()
y_train_s = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
y_test_s  = y_scaler.transform(y_test.reshape(-1, 1)).ravel()

def make_seq(X, y, seq_len):
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32).reshape(-1)
    Xs, ys = [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i:i+seq_len])
        ys.append(y[i+seq_len])
    return torch.tensor(np.stack(Xs)), torch.tensor(np.array(ys))

Xtr_seq, ytr_seq = make_seq(X_train_s, y_train_s, seq_len)
Xte_seq, yte_seq = make_seq(X_test_s,  y_test_s,  seq_len)

train_loader = DataLoader(TensorDataset(Xtr_seq, ytr_seq),
                          batch_size=128, shuffle=True)
test_loader  = DataLoader(TensorDataset(Xte_seq, yte_seq),
                          batch_size=128, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class LSTMRegressor(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True,
                            dropout=0.2 if num_layers > 1 else 0.0)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)          # (B, T, H)
        return self.fc(out[:, -1, :]).squeeze(-1)  # (B,)

model = LSTMRegressor(input_size=Xtr_seq.shape[2]).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.MSELoss()

n_epochs = 10
rmse = None  # will hold the last computed test RMSE

for epoch in range(1, n_epochs+1):
    model.train()
    batch_losses = []

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = crit(model(xb), yb)  # MSE in *scaled* y-space
        loss.backward()
        opt.step()
        batch_losses.append(loss.item())

    train_mse = float(np.mean(batch_losses))

    # evaluate on test set every 5 epochs (and at epoch 1 if you like)
    if epoch % 5 == 0 or epoch == 1:
        model.eval()
        preds_s, trues_s = [], []
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(device)
                preds_s.append(model(xb).cpu().numpy())
                trues_s.append(yb.numpy())

        y_pred_s = np.concatenate(preds_s)
        y_true_s = np.concatenate(trues_s)

        # back to original units
        y_pred = y_scaler.inverse_transform(y_pred_s.reshape(-1, 1)).ravel()
        y_true = y_scaler.inverse_transform(y_true_s.reshape(-1, 1)).ravel()

        rmse = root_mean_squared_error(y_true, y_pred)
        print(
            f"Epoch {epoch:02d}... \t "
            f"Train MSE (scaled): {train_mse:.3f} \t "
            f"Test RMSE: {rmse:.1f}"
        )
    else:
        print(f"Epoch {epoch:02d}... \t Train MSE (scaled): {train_mse:.3f}")

# after training, y_pred / y_true / rmse refer to the last eval (epoch 50)
print(f"\nLSTM RMSE: {rmse:.1f}")

Epoch 01... 	 Train MSE (scaled): 0.318 	 Test RMSE: 1106.0
Epoch 02... 	 Train MSE (scaled): 0.057
Epoch 03... 	 Train MSE (scaled): 0.045
Epoch 04... 	 Train MSE (scaled): 0.041
Epoch 05... 	 Train MSE (scaled): 0.037 	 Test RMSE: 634.8
Epoch 06... 	 Train MSE (scaled): 0.034
Epoch 07... 	 Train MSE (scaled): 0.031
Epoch 08... 	 Train MSE (scaled): 0.030
Epoch 09... 	 Train MSE (scaled): 0.030
Epoch 10... 	 Train MSE (scaled): 0.028 	 Test RMSE: 594.0

LSTM RMSE: 594.0
